# Pixels to Predictions — Phase 1: QLoRA Training & Submission

**This is the main workhorse notebook. Re-run with different settings to improve scores.**

Stages:
1. Setup & load data
2. Load model with QLoRA
3. Train (200 samples → 10% → full)
4. Evaluate on val set
5. Run inference on test set
6. Generate submission.csv

**Zero-shot baseline: 0.5563 | TA baseline: 0.6781**

In [2]:
# ── Cell 1: Mount Drive & Setup ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/kaggle_final_competition")
DATA_DIR = PROJECT_ROOT / "data"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
SUBMISSION_DIR = PROJECT_ROOT / "submissions"

for d in [CHECKPOINT_DIR, SUBMISSION_DIR]:
    d.mkdir(exist_ok=True)

print("Data dir contents:", sorted(os.listdir(DATA_DIR)))
print("Existing checkpoints:", sorted(os.listdir(CHECKPOINT_DIR)))

Mounted at /content/drive
Data dir contents: ['images', 'sample_submission.csv', 'test.csv', 'train.csv', 'val.csv']
Existing checkpoints: ['phase0_val_predictions.csv', 'phase0_zero_shot_results.json']


In [3]:
# ── Cell 2: Install Packages ─────────────────────────────────────────────────
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 104.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.4 MB/s eta 0:00:00


In [4]:
# ── Cell 3: Imports & Config ─────────────────────────────────────────────────
import json
import random
import time
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ═══════════════════════════════════════════════════════════════════════════
# EXPERIMENT CONFIG — CHANGE THESE TO ITERATE
# ═══════════════════════════════════════════════════════════════════════════
MODEL_ID        = "HuggingFaceTB/SmolVLM-500M-Instruct"
IMG_SIZE        = 224
CHOICE_LETTERS  = "ABCDEFGHIJ"

# Training hyperparameters
LORA_RANK       = 16          # try: 8, 16, 32
LORA_ALPHA      = 32          # typically 2x rank
LORA_DROPOUT    = 0.05
LEARNING_RATE   = 2e-4        # try: 5e-5, 1e-4, 2e-4
NUM_EPOCHS      = 1           # try: 1, 2, 3
BATCH_SIZE      = 4           # effective batch on T4
GRAD_ACCUM      = 4           # effective batch = BATCH_SIZE * GRAD_ACCUM = 16
WARMUP_RATIO    = 0.05
MAX_SEQ_LEN     = 512

# Training stage — change this to control data size
# Options: "sanity" (200), "small" (310), "full" (all)
TRAIN_STAGE     = "sanity"

# Experiment name — used for checkpoint naming
EXP_NAME        = f"lora_r{LORA_RANK}_lr{LEARNING_RATE}_{TRAIN_STAGE}_ep{NUM_EPOCHS}"

# ═══════════════════════════════════════════════════════════════════════════

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"\nExperiment: {EXP_NAME}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB

Experiment: lora_r16_lr0.0002_sanity_ep1
Effective batch size: 16


In [5]:
# ── Cell 4: Load Data ────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

# Select training subset based on TRAIN_STAGE
if TRAIN_STAGE == "sanity":
    train_subset = train_df.sample(n=200, random_state=SEED).reset_index(drop=True)
elif TRAIN_STAGE == "small":
    train_subset = train_df.sample(n=310, random_state=SEED).reset_index(drop=True)
elif TRAIN_STAGE == "full":
    train_subset = train_df.copy().reset_index(drop=True)
else:
    raise ValueError(f"Unknown TRAIN_STAGE: {TRAIN_STAGE}")

print(f"Train subset: {len(train_subset)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Training stage: {TRAIN_STAGE}")

Train subset: 200 | Val: 1048 | Test: 1008
Training stage: sanity


In [6]:
# ── Cell 5: Prompt Builder (best from Phase 0) ───────────────────────────────

def build_prompt(row, include_answer=False):
    """
    Default prompt style — winner from Phase 0.
    """
    context_parts = []
    lecture = row.get("lecture", "")
    hint = row.get("hint", "")
    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip())
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())
    context_str = "\n".join(context_parts)

    choices = row["choices"]
    choices_str = "\n".join(f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))

    prompt = "<image>\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        answer_idx = int(row['answer'])
        prompt += f" {CHOICE_LETTERS[answer_idx]}"

    return prompt

print("Prompt builder ready.")
print("\nExample prompt (with answer):")
print(build_prompt(train_subset.iloc[0], include_answer=True))

Prompt builder ready.

Example prompt (with answer):
<image>
Context:
People can use the engineering-design process to develop solutions to problems. One step in the process is testing if a potential solution meets the requirements of the design. How can you determine what a test can show? You need to figure out what was tested and what was measured.
Imagine an engineer needs to design a bridge for a windy location. She wants to make sure the bridge will not move too much in high wind. So, she builds a smaller prototype, or model, of a bridge. Then, she exposes the prototype to high winds and measures how much the bridge moves.
First, identify what was tested. A test can examine one design, or it may compare multiple prototypes to each other. In the test described above, the engineer tested a prototype of a bridge in high wind.
Then, identify what the test measured. One of the criteria for the bridge was that it not move too much in high winds. The test measured how much the prototype 

In [7]:
# ── Cell 6: Load Model with QLoRA ────────────────────────────────────────────
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

print("Loading model with 4-bit quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)

# Prepare for QLoRA
model = prepare_model_for_kbit_training(model)

# Find target modules (attention layers)
# Print all linear layer names to pick LoRA targets
print("\nLinear layers in model:")
target_modules = []
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        # Only grab the short name
        short_name = name.split('.')[-1]
        if short_name not in target_modules:
            target_modules.append(short_name)

print(f"Unique linear layer names: {target_modules}")

# Apply LoRA to attention projection layers
# Common targets: q_proj, k_proj, v_proj, o_proj
lora_targets = ["q_proj", "k_proj", "v_proj", "o_proj"]

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=lora_targets,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

# Print trainable params
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTrainable params: {trainable_params:,} ({trainable_params/1e6:.2f}M)")
print(f"Total params: {total_params:,} ({total_params/1e6:.1f}M)")
print(f"Trainable %: {100 * trainable_params / total_params:.2f}%")

if trainable_params > 5_000_000:
    print("\n⚠️ WARNING: Trainable params exceed 5M limit! Reduce LORA_RANK.")
else:
    print(f"\n✅ Under 5M limit ({5_000_000 - trainable_params:,} params remaining)")

print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

Loading processor...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]


Linear layers in model:
Unique linear layer names: ['k_proj', 'v_proj', 'q_proj', 'out_proj', 'fc1', 'fc2', 'proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj', 'lm_head']

Trainable params: 4,161,536 (4.16M)
Total params: 305,991,872 (306.0M)
Trainable %: 1.36%

✅ Under 5M limit (838,464 params remaining)
VRAM used: 0.6 GB


In [10]:
# ── Cell 7: Training Dataset & Collator ──────────────────────────────────────

class ScienceQATrainDataset(Dataset):
    def __init__(self, df, data_dir, processor, img_size=224, max_len=512):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.processor = processor
        self.img_size = img_size
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Load image
        img = Image.open(self.data_dir / row["image_path"]).convert("RGB")
        img = img.resize((self.img_size, self.img_size), Image.BICUBIC)

        # Build prompt WITH answer for training
        prompt = build_prompt(row, include_answer=True)

        return {"image": img, "text": prompt}


def train_collate_fn(batch):
    """
    Collate function that tokenizes and creates labels for causal LM training.
    Labels = input_ids but with prompt tokens masked to -100 (only train on answer).
    """
    images = [item["image"] for item in batch]
    texts = [item["text"] for item in batch]

    # Get the prompt-only version (without answer) to find where answer starts
    # Answer is always the last token(s) after "Answer:"
    inputs = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=False,  # ← changed from True
    )

    # For causal LM: labels = input_ids, shifted internally by the model
    # We mask everything except the answer token(s) with -100
    labels = inputs["input_ids"].clone()

    # Mask padding tokens
    if "attention_mask" in inputs:
        labels[inputs["attention_mask"] == 0] = -100

    # For each sample, find "Answer:" and mask everything before and including it
    # We only want to compute loss on the answer letter token
    for i in range(len(texts)):
        # Find the position of the answer letter (last non-pad token)
        if "attention_mask" in inputs:
            seq_len = inputs["attention_mask"][i].sum().item()
        else:
            seq_len = labels.shape[1]

        # Mask everything except the last real token (the answer letter)
        # The answer letter is at position seq_len - 1
        labels[i, :seq_len - 1] = -100

    inputs["labels"] = labels
    return inputs


# Create dataset
train_dataset = ScienceQATrainDataset(
    train_subset, DATA_DIR, processor, img_size=IMG_SIZE, max_len=MAX_SEQ_LEN
)
print(f"Training dataset: {len(train_dataset)} samples")

# Quick check: load one sample
sample = train_dataset[0]
print(f"Sample text: {sample['text'][:100]}...")
print(f"Sample image size: {sample['image'].size}")

Training dataset: 200 samples
Sample text: <image>
Context:
People can use the engineering-design process to develop solutions to problems. One...
Sample image size: (224, 224)


In [11]:
# ── Cell 8: Training Loop ────────────────────────────────────────────────────
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=train_collate_fn,
    num_workers=0,  # ← changed from 2
    pin_memory=True,
)

# Optimizer — only trainable (LoRA) params
optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=0.01,
)

total_steps = len(train_loader) * NUM_EPOCHS // GRAD_ACCUM
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps)

print(f"Total batches per epoch: {len(train_loader)}")
print(f"Total optimization steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")
print(f"\nStarting training...")
print(f"={'='*60}")

# Training loop with progress bars and checkpointing
model.train()
training_log = []
global_step = 0
best_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    epoch_losses = []
    optimizer.zero_grad()

    pbar = tqdm(
        enumerate(train_loader),
        total=len(train_loader),
        desc=f"Epoch {epoch+1}/{NUM_EPOCHS}",
        leave=True,
    )

    for batch_idx, batch in pbar:
        # Move to device
        batch = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in batch.items()}

        # Forward pass
        outputs = model(**batch)
        loss = outputs.loss / GRAD_ACCUM
        loss.backward()

        epoch_losses.append(outputs.loss.item())

        # Gradient accumulation step
        if (batch_idx + 1) % GRAD_ACCUM == 0 or (batch_idx + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

        # Update progress bar
        avg_loss = np.mean(epoch_losses[-20:])  # rolling avg of last 20 batches
        pbar.set_postfix({
            "loss": f"{outputs.loss.item():.4f}",
            "avg_loss": f"{avg_loss:.4f}",
            "lr": f"{scheduler.get_last_lr()[0]:.2e}",
            "VRAM": f"{torch.cuda.memory_allocated()/1e9:.1f}GB",
        })

        # Log every 10 batches
        if (batch_idx + 1) % 10 == 0:
            training_log.append({
                "epoch": epoch + 1,
                "batch": batch_idx + 1,
                "loss": outputs.loss.item(),
                "avg_loss": avg_loss,
                "lr": scheduler.get_last_lr()[0],
            })

        # Checkpoint every 25% of epoch
        checkpoint_interval = max(len(train_loader) // 4, 1)
        if (batch_idx + 1) % checkpoint_interval == 0:
            mid_ckpt_path = CHECKPOINT_DIR / f"{EXP_NAME}_epoch{epoch+1}_batch{batch_idx+1}"
            model.save_pretrained(str(mid_ckpt_path))
            print(f"\n  📌 Mid-checkpoint saved: {mid_ckpt_path.name}")

    # End of epoch summary
    epoch_avg_loss = np.mean(epoch_losses)
    print(f"\nEpoch {epoch+1} complete — Avg loss: {epoch_avg_loss:.4f}")

    # Save epoch checkpoint
    epoch_ckpt_path = CHECKPOINT_DIR / f"{EXP_NAME}_epoch{epoch+1}_final"
    model.save_pretrained(str(epoch_ckpt_path))
    print(f"📌 Epoch checkpoint saved: {epoch_ckpt_path.name}")

    if epoch_avg_loss < best_loss:
        best_loss = epoch_avg_loss
        best_ckpt_path = CHECKPOINT_DIR / f"{EXP_NAME}_best"
        model.save_pretrained(str(best_ckpt_path))
        print(f"⭐ New best model saved: {best_ckpt_path.name}")

# Save training log
log_path = CHECKPOINT_DIR / f"{EXP_NAME}_training_log.json"
with open(log_path, "w") as f:
    json.dump(training_log, f, indent=2)
print(f"\nTraining log saved: {log_path.name}")
print(f"\n{'='*60}")
print(f"TRAINING COMPLETE")
print(f"Best loss: {best_loss:.4f}")
print(f"{'='*60}")

Total batches per epoch: 50
Total optimization steps: 12
Warmup steps: 0

Starting training...


Epoch 1/1:   0%|          | 0/50 [00:00<?, ?it/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)



  📌 Mid-checkpoint saved: lora_r16_lr0.0002_sanity_ep1_epoch1_batch12

  📌 Mid-checkpoint saved: lora_r16_lr0.0002_sanity_ep1_epoch1_batch24

  📌 Mid-checkpoint saved: lora_r16_lr0.0002_sanity_ep1_epoch1_batch36

  📌 Mid-checkpoint saved: lora_r16_lr0.0002_sanity_ep1_epoch1_batch48

Epoch 1 complete — Avg loss: 1.1003
📌 Epoch checkpoint saved: lora_r16_lr0.0002_sanity_ep1_epoch1_final
⭐ New best model saved: lora_r16_lr0.0002_sanity_ep1_best

Training log saved: lora_r16_lr0.0002_sanity_ep1_training_log.json

TRAINING COMPLETE
Best loss: 1.1003


In [12]:
# Check loss trend
import json
log_path = CHECKPOINT_DIR / f"{EXP_NAME}_training_log.json"
with open(log_path) as f:
    log = json.load(f)

print("Loss trend (every 10 batches):")
for entry in log:
    print(f"  Batch {entry['batch']:>3}: loss={entry['loss']:.4f}  avg={entry['avg_loss']:.4f}")

Loss trend (every 10 batches):
  Batch  10: loss=1.2801  avg=1.5118
  Batch  20: loss=1.5078  avg=1.3903
  Batch  30: loss=1.4096  avg=1.0280
  Batch  40: loss=1.1650  avg=0.9230
  Batch  50: loss=0.3949  avg=0.9669


In [13]:
# ── Cell 9: Validation — Log-Likelihood Scoring ──────────────────────────────

# Pre-compute choice token IDs
choice_token_ids = {}
for letter in CHOICE_LETTERS[:5]:
    tokens = processor.tokenizer.encode(letter, add_special_tokens=False)
    choice_token_ids[letter] = tokens[0]


def evaluate_log_likelihood(df, prompt_fn, batch_size=16, desc="Evaluating"):
    """
    Evaluate using log-likelihood scoring with progress bar and
    intermediate accuracy reporting.
    """
    model.eval()
    predictions = []
    correct = 0
    total = 0

    pbar = tqdm(range(0, len(df), batch_size), desc=desc, leave=True)

    for start in pbar:
        end = min(start + batch_size, len(df))
        chunk = df.iloc[start:end]

        # Build prompts and load images
        prompts = []
        images = []
        num_choices_list = []

        for _, row in chunk.iterrows():
            prompts.append(prompt_fn(row, include_answer=False))
            img = Image.open(DATA_DIR / row["image_path"]).convert("RGB")
            img = img.resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC)
            images.append(img)
            num_choices_list.append(int(row["num_choices"]))

        # Tokenize
        inputs = processor(
            text=prompts,
            images=images,
            return_tensors="pt",
            padding=True,
            truncation=True,
        )
        inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

        # Forward pass
        with torch.inference_mode():
            outputs = model(**inputs)

        logits = outputs.logits

        # Find last non-pad position
        if "attention_mask" in inputs:
            seq_lengths = inputs["attention_mask"].sum(dim=1) - 1
        else:
            seq_lengths = torch.tensor([logits.shape[1] - 1] * logits.shape[0])

        for i in range(len(chunk)):
            last_pos = seq_lengths[i].item()
            last_logits = logits[i, int(last_pos), :]
            log_probs = F.log_softmax(last_logits, dim=-1)

            n_choices = num_choices_list[i]
            choice_scores = []
            for j in range(n_choices):
                letter = CHOICE_LETTERS[j]
                token_id = choice_token_ids[letter]
                choice_scores.append(log_probs[token_id].item())

            pred = int(np.argmax(choice_scores))
            predictions.append(pred)

            # Track running accuracy
            actual = int(chunk.iloc[i]["answer"]) if "answer" in chunk.columns else -1
            if actual >= 0:
                total += 1
                if pred == actual:
                    correct += 1

        # Update progress bar with running accuracy
        if total > 0:
            pbar.set_postfix({"acc": f"{correct/total:.4f} ({correct}/{total})"})

        # Free GPU memory
        del inputs, outputs, logits
        torch.cuda.empty_cache()

    final_acc = correct / total if total > 0 else 0
    return predictions, final_acc


print("Evaluation function ready.")

Evaluation function ready.


In [17]:
# ── Force free all training memory ───────────────────────────────────────────
import gc

# Delete optimizer and scheduler
del optimizer, scheduler, train_loader, train_dataset
gc.collect()
torch.cuda.empty_cache()

print(f"VRAM after full cleanup: {torch.cuda.memory_allocated()/1e9:.1f} GB")

VRAM after full cleanup: 5.2 GB


In [18]:
# ── Cell 10: Run Validation ──────────────────────────────────────────────────
print(f"Evaluating on full val set ({len(val_df)} samples)...")
print(f"Model: {EXP_NAME}")

start_time = time.time()
val_preds, val_acc = evaluate_log_likelihood(
    val_df, build_prompt, batch_size=4, desc="Val Eval"
)
elapsed = time.time() - start_time

print(f"\n{'='*60}")
print(f"VALIDATION RESULTS — {EXP_NAME}")
print(f"{'='*60}")
print(f"Val Accuracy:     {val_acc:.4f}")
print(f"Zero-shot:        0.5563")
print(f"TA Baseline:      0.6781")
print(f"Improvement:      {val_acc - 0.5563:+.4f} vs zero-shot")
print(f"Gap to TA:        {0.6781 - val_acc:.4f}")
print(f"Time:             {elapsed:.1f}s")

# Per-category breakdown
val_df_eval = val_df.copy()
val_df_eval["pred"] = val_preds
val_df_eval["correct"] = val_df_eval["pred"] == val_df_eval["answer"].astype(int)

print(f"\nAccuracy by SUBJECT:")
print(val_df_eval.groupby("subject")["correct"].mean().sort_values(ascending=False).to_string())

print(f"\nAccuracy by NUM_CHOICES:")
print(val_df_eval.groupby("num_choices")["correct"].mean().sort_values(ascending=False).to_string())

# Save val results
val_results = {
    "experiment": EXP_NAME,
    "val_accuracy": val_acc,
    "zero_shot_baseline": 0.5563,
    "ta_baseline": 0.6781,
    "time_seconds": elapsed,
    "per_subject": val_df_eval.groupby("subject")["correct"].mean().to_dict(),
    "per_num_choices": val_df_eval.groupby("num_choices")["correct"].mean().to_dict(),
    "config": {
        "lora_rank": LORA_RANK,
        "lora_alpha": LORA_ALPHA,
        "lr": LEARNING_RATE,
        "epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "train_stage": TRAIN_STAGE,
        "train_samples": len(train_subset),
    }
}

results_path = CHECKPOINT_DIR / f"{EXP_NAME}_val_results.json"
with open(results_path, "w") as f:
    json.dump(val_results, f, indent=2)
print(f"\nResults saved: {results_path.name}")

Evaluating on full val set (1048 samples)...
Model: lora_r16_lr0.0002_sanity_ep1


Val Eval:   0%|          | 0/262 [00:00<?, ?it/s]


VALIDATION RESULTS — lora_r16_lr0.0002_sanity_ep1
Val Accuracy:     0.5334
Zero-shot:        0.5563
TA Baseline:      0.6781
Improvement:      -0.0229 vs zero-shot
Gap to TA:        0.1447
Time:             1925.9s

Accuracy by SUBJECT:
subject
social science      0.551440
natural science     0.535393
language science    0.321429

Accuracy by NUM_CHOICES:
num_choices
2    0.680328
4    0.531746
3    0.490157
5    0.227273

Results saved: lora_r16_lr0.0002_sanity_ep1_val_results.json


In [ ]:
# ── Cell 11: Test Inference & Submission ─────────────────────────────────────
# Only run this when you're happy with val accuracy!

print(f"Running test inference ({len(test_df)} samples)...")

start_time = time.time()
test_preds, _ = evaluate_log_likelihood(
    test_df, build_prompt, batch_size=16, desc="Test Inference"
)
elapsed = time.time() - start_time

print(f"\nTest inference complete in {elapsed:.1f}s")
print(f"Predictions: {len(test_preds)}")

# Create submission
submission = pd.DataFrame({
    "id": test_df["id"],
    "answer": test_preds,
})

# Verify format against sample submission
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
assert len(submission) == len(sample_sub), f"Length mismatch: {len(submission)} vs {len(sample_sub)}"
assert list(submission.columns) == list(sample_sub.columns), f"Column mismatch"
assert submission["id"].tolist() == sample_sub["id"].tolist(), "ID mismatch — reordering needed"

# Save submission
sub_filename = f"submission_{EXP_NAME}.csv"
sub_path = SUBMISSION_DIR / sub_filename
submission.to_csv(sub_path, index=False)

# Also save as submission.csv (latest)
submission.to_csv(SUBMISSION_DIR / "submission.csv", index=False)

print(f"\n{'='*60}")
print(f"SUBMISSION READY")
print(f"{'='*60}")
print(f"Saved to: {sub_path}")
print(f"Also saved as: submission.csv")
print(f"\nPrediction distribution:")
print(submission["answer"].value_counts().sort_index())
print(f"\nFirst 5 rows:")
print(submission.head())
print(f"\n→ Upload {sub_path.name} to Kaggle!")

In [ ]:
# ── Cell 12: Load a Saved Checkpoint (for resuming) ──────────────────────────
# Use this cell to load a previously saved LoRA adapter
# Uncomment and modify the path to resume from a checkpoint

# from peft import PeftModel
#
# CHECKPOINT_TO_LOAD = "lora_r16_lr0.0002_sanity_ep1_best"
#
# # First load base model (same as Cell 6 but without LoRA)
# base_model = AutoModelForVision2Seq.from_pretrained(
#     MODEL_ID,
#     quantization_config=bnb_config,
#     device_map="auto",
#     low_cpu_mem_usage=True,
# )
#
# # Load LoRA adapter
# ckpt_path = str(CHECKPOINT_DIR / CHECKPOINT_TO_LOAD)
# model = PeftModel.from_pretrained(base_model, ckpt_path)
# model.eval()
# print(f"Loaded checkpoint: {CHECKPOINT_TO_LOAD}")
# print(f"Now run Cell 10 (validation) or Cell 11 (test submission)")

## How to Iterate

1. **First run:** Keep `TRAIN_STAGE = "sanity"` (200 samples). Check that loss decreases.
2. **If loss decreases:** Change to `TRAIN_STAGE = "small"` (310 samples). Run Cell 8 + 10.
3. **If val accuracy improves:** Change to `TRAIN_STAGE = "full"`. Run Cell 8 + 10.
4. **To try different hyperparams:** Change `LORA_RANK`, `LEARNING_RATE`, `NUM_EPOCHS` in Cell 3.
5. **To resume from checkpoint:** Use Cell 12.
6. **When happy with val accuracy:** Run Cell 11 to generate submission.

### Hyperparameter Ideas
| Setting | Conservative | Moderate | Aggressive |
|---|---|---|---|
| LORA_RANK | 8 | 16 | 32 |
| LEARNING_RATE | 5e-5 | 1e-4 | 2e-4 |
| NUM_EPOCHS | 1 | 2 | 3 |
| LORA targets | qkv only | qkvo | qkvo + mlp |